In [0]:
dbutils.widgets.text("catalogo", "mlops", "Catálogo")
dbutils.widgets.text("esquema", "frecuencia", "Esquema")
dbutils.widgets.text("experimento", "/Users/santiagopadu@gmail.com/frecuencia_rc", "Experimento")
dbutils.widgets.text("alphas", "0.0001,0.001,0.01,0.1", "Alphas")

catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
experimento = dbutils.widgets.get("experimento")
alphas = [float(a) for a in dbutils.widgets.get("alphas").split(",")]

In [0]:
tabla_train = f"{catalogo}.{esquema}.polizas_train"
tabla_test = f"{catalogo}.{esquema}.polizas_test"

def version_delta(tabla):
    return spark.sql(f"DESCRIBE HISTORY {tabla} LIMIT 1").collect()[0]["version"]

version_train = version_delta(tabla_train)
version_test = version_delta(tabla_test)

train = spark.sql(f"SELECT * FROM {tabla_train} VERSION AS OF {version_train}").toPandas()
test = spark.sql(f"SELECT * FROM {tabla_test} VERSION AS OF {version_test}").toPandas()

print(len(train), version_train, len(test), version_test)

In [0]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer, FunctionTransformer, StandardScaler
from sklearn.linear_model import PoissonRegressor

categoricas = ["Area", "VehBrand", "VehGas", "Region", "VehPower"]
en_tramos = ["VehAge", "DrivAge"]
directas = ["BonusMalus"]
logaritmicas = ["Density"]
variables = categoricas + en_tramos + directas + logaritmicas

def crear_pipeline(alpha):
    preproceso = ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), categoricas),
        ("tramos", KBinsDiscretizer(n_bins=10, encode="onehot", strategy="quantile", subsample=None), en_tramos),
        ("directas", "passthrough", directas),
        ("logaritmo", make_pipeline(FunctionTransformer(np.log), StandardScaler()), logaritmicas),
    ])
    return Pipeline([
        ("preproceso", preproceso),
        ("glm", PoissonRegressor(alpha=alpha, solver="newton-cholesky", max_iter=300)),
    ])

X_train, X_test = train[variables], test[variables]
y_train = train["ClaimNb"] / train["Exposure"]
y_test = test["ClaimNb"] / test["Exposure"]
w_train, w_test = train["Exposure"], test["Exposure"]

print(crear_pipeline(alphas[0])["preproceso"].fit_transform(X_train).shape)

In [0]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_poisson_deviance
from mlflow.models import infer_signature

mlflow.sklearn.autolog(disable=True)
mlflow.set_experiment(experimento)

datos_train = mlflow.data.load_delta(table_name=tabla_train, version=version_train)
datos_test = mlflow.data.load_delta(table_name=tabla_test, version=version_test)

def grafico_deciles(mu, df):
    g = df.assign(esperados=mu * df["Exposure"], decil=pd.qcut(mu, 10, labels=False, duplicates="drop") + 1)
    r = g.groupby("decil")[["ClaimNb", "esperados", "Exposure"]].sum()
    fig, ax = plt.subplots()
    ax.plot(r.index, r["ClaimNb"] / r["Exposure"], marker="o", label="Observada")
    ax.plot(r.index, r["esperados"] / r["Exposure"], marker="o", label="Predicha")
    ax.set_xlabel("Decil de frecuencia predicha")
    ax.set_ylabel("Siniestros por año")
    ax.legend()
    return fig

for alpha in alphas:
    with mlflow.start_run(run_name=f"glm_alpha_{alpha}"):
        modelo = crear_pipeline(alpha).fit(X_train, y_train, glm__sample_weight=w_train)
        mu_train = modelo.predict(X_train)
        mu_test = modelo.predict(X_test)

        mlflow.log_params({
            "alpha": alpha,
            "tabla_train": tabla_train,
            "version_train": version_train,
            "tabla_test": tabla_test,
            "version_test": version_test,
        })
        mlflow.log_input(datos_train, context="training")
        mlflow.log_input(datos_test, context="test")
        mlflow.log_metrics({
            "devianza_train": mean_poisson_deviance(y_train, mu_train, sample_weight=w_train),
            "devianza_test": mean_poisson_deviance(y_test, mu_test, sample_weight=w_test),
            "ae_train": train["ClaimNb"].sum() / (mu_train * w_train).sum(),
            "ae_test": test["ClaimNb"].sum() / (mu_test * w_test).sum(),
        })

        fig = grafico_deciles(mu_test, test)
        mlflow.log_figure(fig, "deciles_test.png")
        plt.close(fig)

        mlflow.sklearn.log_model(
            modelo,
            name="modelo",
            signature=infer_signature(X_train, mu_train),
            input_example=X_train.head(5),
        )